# 📬 Email Spam Classifier
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/email-spam-classifier/training/notebook.ipynb)

Decide whether an email is **spam** or **ham** (legitimate). A pretrained DistilBERT encoder
(uncased) gets a new one-logit head, and its last 2 transformer blocks are fine-tuned on
~23.6k Enron-Spam emails. A TF-IDF + logistic-regression baseline is trained next to it for
comparison. The notebook exports the Hugging Face model folder that the
[Space](https://huggingface.co/spaces/shalev396/email-spam-classifier) serves.

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "email-spam-classifier"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as ShowImage, display

from src import data_setup, engine, export, model_builder, utils   # importing src puts ../model on sys.path
from src.config import Config
import model as M                                                    # ../model/model.py, what the Space runs

print(utils.lib_versions("torch", "transformers", "tokenizers", "sklearn", "huggingface_hub"))
torch_device = torch.device(device)

## 2. Config
Every hyperparameter lives in `src/config.py`. The defaults are the recipe of the deployed model:
max 256 tokens, last 2 blocks unfrozen, AdamW (head 5e-4, encoder 2e-5, weight decay 0.01), 10% linear
warmup, `BCEWithLogitsLoss(pos_weight = n_ham / n_spam)`, batch 32, up to 5 epochs with early stopping
(patience 2) on validation loss, seed 42. `SMOKE_TEST=1` shrinks the run to 128 train / 64 val / 64 test
emails, 64 tokens and 1 epoch, and exports to `outputs/smoke/model` instead of `../model`.

In [ ]:
os.environ.setdefault("SMOKE_TEST", "0")   # "1" = quick smoke run; a SMOKE_TEST set outside the notebook wins
PUSH_TO_HUB = False                         # upload the exported model folder to the Hub (section 8)

cfg = Config()
print("checkpoint:", cfg.checkpoint_dir)
print("export to: ", cfg.model_dir)
cfg

## 3. Data
[Enron-Spam](https://huggingface.co/datasets/SetFit/enron_spam) (Metsis et al., 2006; Hub copy by SetFit):
real Enron employee mailboxes (ham) mixed with spam from several spam traps. Each `text` is subject + body.
`data_setup.load_splits` downloads it once (Hub `datasets` loader, raw JSONL files as fallback), cleans every
email with `model.clean_text` (the exact function used at inference), drops empty texts and makes a
stratified 70/15/15 split cached as `training/data/enron_{train,val,test}.csv`.

In [ ]:
train_df, val_df, test_df = data_setup.load_splits(cfg)
splits = {"train": train_df, "val": val_df, "test": test_df}
pos_weight = data_setup.compute_pos_weight(train_df)
print(f"pos_weight (ham / spam on train) = {pos_weight:.3f}")
data_setup.summary(splits)

In [ ]:
words = train_df["text"].str.split().str.len().clip(upper=1000)
fig, ax = plt.subplots(figsize=(7, 3))
for label, color in ((0, "#2a78d6"), (1, "#eb6834")):
    ax.hist(words[train_df["label"] == label], bins=50, alpha=0.6, color=color, label=M.LABELS[label])
ax.set(title="Words per email (train, clipped at 1000)", xlabel="words", ylabel="emails")
ax.legend(frameon=False)
plt.show()
for label in (0, 1):
    print(M.LABELS[label], "->", train_df.loc[train_df["label"] == label, "text"].iloc[0][:200], "...")

## 4. Load model
`model_builder.build_model` creates `model.SpamClassifier` from `../model/model.py` with the pretrained
DistilBERT weights (`pretrained=True`), freezes the encoder and unfreezes its last `unfreeze_last_n` blocks.
The encoder's config is stored in `config.json`, so inference rebuilds it offline (`pretrained=False`).

In [ ]:
tokenizer = model_builder.load_tokenizer(cfg)
net = model_builder.build_model(cfg)
print(net.classifier)
print(f"parameters: {utils.count_params(net):,} total, {model_builder.trainable_params(net):,} trainable "
      f"(last {cfg.unfreeze_last_n} blocks + head)")

## 5. Training
Two experiments on the same train split:
1. **Baseline**: TF-IDF (word 1-2-grams) + class-balanced logistic regression, under a minute on a CPU.
2. **DistilBERT fine-tune** (the deployed model): `engine.fit` with early stopping on validation loss;
   the best epoch is restored and saved to `cfg.checkpoint_dir` as a loadable model folder.

A full DistilBERT run is a GPU job: the original run (5 epochs) used a CUDA GPU with mixed precision.
On a CPU expect several hours. The smoke run takes about a minute.

In [ ]:
baseline = engine.train_baseline(cfg, train_df)
print(f"baseline trained in {baseline.train_time_s}s, {len(baseline.named_steps['tfidf'].vocabulary_):,} features")

loaders = data_setup.create_dataloaders(cfg, splits, tokenizer, torch_device)
history = engine.fit(net, loaders, cfg, torch_device, pos_weight)
engine.save_checkpoint(net, tokenizer, cfg.checkpoint_dir)
display(ShowImage(filename=str(export.plot_history(history, cfg.run_dir / "training_curves.png"))))

## 6. Evaluation
The fine-tuned model is scored on the held-out test split through `model.load(checkpoint).predict_proba`,
the same code path as the Space and the endpoint. Next to it: the baseline, and the model currently deployed
in `../model` (numbers from its `metrics.json`).

In [ ]:
predictor = M.load(cfg.checkpoint_dir, device)
y_true = test_df["label"].to_numpy()
y_prob = engine.predict_texts(predictor, test_df["text"].tolist(), cfg.eval_batch_size)
b_prob = engine.baseline_proba(baseline, test_df["text"])

DEPLOYED, BASELINE = "DistilBERT fine-tuned", "TF-IDF + logistic regression"
scores = engine.binary_metrics(y_true, y_prob, cfg.threshold)
comparison = {DEPLOYED: scores, BASELINE: engine.binary_metrics(y_true, b_prob, 0.5)}
table = pd.DataFrame(comparison).T
table.loc["model/ now (from its metrics.json)"] = pd.Series(utils.load_json(utils.MODEL_DIR / "metrics.json")["metrics"])
table.round(4)

In [ ]:
cm = engine.confusion(y_true, y_prob, cfg.threshold)
print("confusion matrix (rows = true ham/spam, cols = predicted):\n", cm)
display(ShowImage(filename=str(export.plot_confusion_matrix(y_true, y_prob, cfg.run_dir / "confusion_matrix.png")), width=420))
display(ShowImage(filename=str(export.plot_roc_pr({DEPLOYED: (y_true, y_prob), BASELINE: (y_true, b_prob)},
                                                  cfg.run_dir / "roc_pr_curves.png"))))
display(ShowImage(filename=str(export.plot_comparison(comparison, cfg.run_dir / "comparison.png"))))

## 7. Inference
Exactly what the Space and the endpoint run: `model.load(dir, device).predict(text)` takes
`"Subject: <subject>\n\n<body>"` and returns `{"spam": p, "ham": 1 - p}`.

In [ ]:
for path in sorted((utils.SPACE_DIR / "examples").glob("*.txt")):
    probs = predictor.predict(path.read_text(encoding="utf-8"))
    print(f"{path.name:24s} P(spam) = {probs['spam']:.4f} -> {'spam' if probs['spam'] >= predictor.threshold else 'ham'}")
predictor.predict(M.compose_email("Quick question", "Are we still on for lunch tomorrow at noon?"))

## 8. Export
`export.export` copies the checkpoint (`model.safetensors`, `config.json`, tokenizer) into `cfg.model_dir`
and writes `metrics.json`, the plots in `assets/` and the refreshed model card. **With `SMOKE_TEST=0` this
overwrites `../model`** (the deployed weights, card metrics and plots); smoke runs go to `outputs/smoke/model`.
To publish, set `PUSH_TO_HUB = True` in section 2. The token comes from `hf auth login` (local) or an
`HF_TOKEN` secret / environment variable (Colab: the Secrets panel). Smoke runs are never uploaded.

In [ ]:
metrics = export.build_metrics(
    scores, data={"n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
                  "spam_share_test": round(float(y_true.mean()), 4), "n_classes": 2},
    params=utils.count_params(net), train_time_s=history["train_time_s"], device=torch_device.type,
    smoke=cfg.smoke, source=f"retrained {utils.today()} with training/ ({torch_device.type})",
    comparison=comparison, history=history, confusion=cm, threshold=cfg.threshold)
metrics["trainable_params"] = model_builder.trainable_params(net)
export.export(cfg.checkpoint_dir, cfg.model_dir, metrics, y_true=y_true, y_prob=y_prob,
              curves={DEPLOYED: (y_true, y_prob), BASELINE: (y_true, b_prob)})

for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{path.relative_to(cfg.model_dir).as_posix():28s} {path.stat().st_size / 1e6:8.2f} MB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))